# 🧠 Memory Lane — RAG Knowledge Explorer (notebook edition)

Ask questions about your documents and get answers **quoted from the sources, with citations**, then watch the
retrieved evidence light up in an interactive **knowledge graph**.

**Pipeline:** ingestion → recursive chunking → Hugging Face embeddings → FAISS → hybrid search (semantic + BM25)
→ cross-encoder reranking → grounded answer with evidence indicators → stale-context checks → incremental re-indexing.

**How to use:** run the cells top to bottom (*Run All*). The first run downloads two small Hugging Face models
(~130 MB). Sample documents are created in a `knowledge_base/` folder next to this notebook — put your own
`.md`, `.txt` or `.pdf` files there and press **Sync** in the app cell.

## 1 · Install & imports

In [3]:
import faiss
import sentence_transformers
import langchain_huggingface
import streamlit

print("All imports successful! Memory Lane dependencies are ready.")

W1101 14:18:26.946000 20900 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


All imports successful! Memory Lane dependencies are ready.


In [ ]:
# Run once. (Skip if the packages are already installed.)
%pip install -q langchain-core langchain-text-splitters langchain-huggingface sentence-transformers faiss-cpu rank-bm25 pypdf pandas numpy ipywidgets


In [4]:
from __future__ import annotations


import hashlib, html, json, re
from dataclasses import asdict, dataclass, field
from datetime import datetime
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from langchain_text_splitters import RecursiveCharacterTextSplitter

import ipywidgets as widgets
from IPython.display import HTML, clear_output, display


## 2 · Ingestion & recursive chunking
Reads `.md` / `.txt` / `.pdf` files (optional front matter: `title`, `last_updated`, `valid_until`, `superseded_by`) and
splits them with LangChain's `RecursiveCharacterTextSplitter` — sections → paragraphs → sentences, with overlap.

In [5]:
from __future__ import annotations

EXTENSIONS = {".md", ".txt", ".pdf"}
SEPARATORS = ["\n## ", "\n### ", "\n\n", "\n", ". ", " ", ""]   # sections > paragraphs > lines > sentences


@dataclass
class Chunk:
    id: int            # FAISS id
    doc: str           # path relative to the knowledge-base folder
    title: str
    n: int             # position within the document
    text: str
    heading: str | None = None
    page: int | None = None


def parse_front_matter(text: str) -> tuple[dict, str]:
    """Flat ``key: value`` block between two ``---`` lines at the top of a file."""
    m = re.match(r"^---[ \t]*\n(.*?)\n---[ \t]*\n?", text, re.S)
    if not m:
        return {}, text
    meta = {}
    for line in m.group(1).splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            meta[k.strip().lower()] = v.strip().strip("\"'")
    return meta, text[m.end():]


def load_file(path: Path) -> tuple[str, dict, list[tuple[int | None, str]]]:
    """Return (title, metadata, [(page, text), ...])."""
    if path.suffix.lower() == ".pdf":
        from pypdf import PdfReader

        pages = [(i, p.extract_text() or "") for i, p in enumerate(PdfReader(str(path)).pages, 1)]
        return path.stem.replace("_", " ").title(), {}, pages
    meta, body = parse_front_matter(path.read_text(encoding="utf-8", errors="replace"))
    h1 = re.search(r"^#\s+(.+)$", body, re.M)
    title = meta.get("title") or (h1.group(1).strip() if h1 else path.stem.replace("_", " ").title())
    return title, meta, [(None, body)]


def chunk_pages(doc: str, title: str, pages, size: int, overlap: int) -> list[Chunk]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=overlap, separators=SEPARATORS, add_start_index=True
    )
    chunks: list[Chunk] = []
    for page, text in pages:
        heads = [(m.start(), m.group(2)) for m in re.finditer(r"^(#{1,4})\s+(.+)$", text, re.M)]
        for d in splitter.create_documents([text]):
            body = d.page_content.strip()
            if len(body) < 20:
                continue
            start, heading = d.metadata["start_index"], None
            for pos, h in heads:
                if pos > start + 2:
                    break
                heading = h
            chunks.append(Chunk(0, doc, title, len(chunks), body, heading, page))
    return chunks


def _sha(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def _date(value: str | None) -> datetime | None:
    for fmt in ("%Y-%m-%d", "%Y/%m/%d", "%d %B %Y", "%B %d, %Y"):
        try:
            return datetime.strptime((value or "").strip(), fmt)
        except ValueError:
            pass
    return None


## 3 · Knowledge base: embeddings, FAISS index, re-indexing, freshness
`KnowledgeBase` keeps a FAISS `IndexIDMap2(IndexFlatIP)` (cosine similarity). Each document's vectors are stored under
their own ids, so **Sync** re-embeds only new or changed files and removes deleted ones. `freshness()` flags documents
that are `stale`, `expired`, `superseded`, `modified` or `missing`.

In [6]:
from __future__ import annotations

class KnowledgeBase:
    """FAISS-backed index of a folder of documents.

    ``embeddings`` is any LangChain ``Embeddings`` object (e.g. ``HuggingFaceEmbeddings``).
    Each document's vectors are stored under their own ids, so re-indexing a changed file
    only touches that file (``IndexIDMap2`` supports removal).
    """

    def __init__(self, kb_dir, index_dir, embeddings, model_name: str,
                 chunk_size: int = 700, chunk_overlap: int = 120, stale_days: int = 365):
        self.kb_dir, self.index_dir = Path(kb_dir), Path(index_dir)
        self.emb, self.stale_days = embeddings, stale_days
        self.signature = {"model": model_name, "chunk_size": chunk_size, "chunk_overlap": chunk_overlap}
        self.chunks: dict[int, Chunk] = {}
        self.docs: dict[str, dict] = {}
        self.index: faiss.Index | None = None
        self.next_id, self.version = 1, 0
        self._hash_cache: dict[str, tuple] = {}
        self._matrix: tuple | None = None
        self._load()

    # ---- embeddings
    def _norm(self, v) -> np.ndarray:
        v = np.asarray(v, dtype="float32")
        return v / np.clip(np.linalg.norm(v, axis=-1, keepdims=True), 1e-9, None)

    def embed_query(self, text: str) -> np.ndarray:
        return self._norm(self.emb.embed_query(text))

    # ---- persistence
    def _load(self) -> None:
        f_idx, f_json = self.index_dir / "index.faiss", self.index_dir / "store.json"
        if not (f_idx.exists() and f_json.exists()):
            return
        data = json.loads(f_json.read_text(encoding="utf-8"))
        if data["signature"] != self.signature:      # different model / chunking -> rebuild from scratch
            return
        self.index = faiss.read_index(str(f_idx))
        self.chunks = {c["id"]: Chunk(**c) for c in data["chunks"]}
        self.docs, self.next_id = data["docs"], data["next_id"]

    def save(self) -> None:
        self.index_dir.mkdir(parents=True, exist_ok=True)
        if self.index is not None:
            faiss.write_index(self.index, str(self.index_dir / "index.faiss"))
        (self.index_dir / "store.json").write_text(json.dumps({
            "signature": self.signature, "next_id": self.next_id, "docs": self.docs,
            "chunks": [asdict(c) for c in self.chunks.values()],
        }), encoding="utf-8")

    # ---- indexing
    def _remove(self, doc: str) -> None:
        ids = self.docs.pop(doc)["chunk_ids"]
        if ids and self.index is not None:
            self.index.remove_ids(faiss.IDSelectorBatch(np.asarray(ids, dtype="int64")))
        for i in ids:
            self.chunks.pop(i, None)

    def _add(self, doc: str, title: str, meta: dict, digest: str, chunks: list[Chunk]) -> None:
        ids = list(range(self.next_id, self.next_id + len(chunks)))
        self.next_id += len(chunks)
        if chunks:
            vecs = self._norm(self.emb.embed_documents([c.text for c in chunks]))
            if self.index is None:
                self.index = faiss.IndexIDMap2(faiss.IndexFlatIP(vecs.shape[1]))
            self.index.add_with_ids(vecs, np.asarray(ids, dtype="int64"))
        for c, i in zip(chunks, ids):
            c.id = i
            self.chunks[i] = c
        self.docs[doc] = {"title": title, "meta": meta, "hash": digest, "chunk_ids": ids,
                          "indexed_at": datetime.now().timestamp()}

    def sync(self, rebuild: bool = False) -> dict:
        """Index new files, re-index changed ones, drop deleted ones. ``rebuild`` starts over."""
        if rebuild:
            self.chunks, self.docs, self.index, self.next_id = {}, {}, None, 1
        files = {}
        for p in sorted(self.kb_dir.rglob("*")):
            rel = p.relative_to(self.kb_dir)
            if p.is_file() and p.suffix.lower() in EXTENSIONS and not any(s.startswith(".") for s in rel.parts):
                files[rel.as_posix()] = p
        report = {"added": [], "updated": [], "removed": []}
        for doc in [d for d in self.docs if d not in files]:
            self._remove(doc)
            report["removed"].append(doc)
        for doc, path in files.items():
            digest, known = _sha(path), self.docs.get(doc)
            if known and known["hash"] == digest:
                continue
            title, meta, pages = load_file(path)
            chunks = chunk_pages(doc, title, pages, self.signature["chunk_size"], self.signature["chunk_overlap"])
            if known:
                self._remove(doc)
            self._add(doc, title, meta, digest, chunks)
            report["updated" if known else "added"].append(doc)
        if any(report.values()) or rebuild:
            self.version += 1
            self.save()
        return report

    def matrix(self) -> tuple[list[int], np.ndarray]:
        """All (chunk id, vector) pairs, cached per index version."""
        if self._matrix is None or self._matrix[0] != self.version:
            ids = sorted(self.chunks)
            vecs = np.vstack([self.index.reconstruct(i) for i in ids]) if ids else np.zeros((0, 1), "float32")
            self._matrix = (self.version, ids, vecs)
        return self._matrix[1], self._matrix[2]

    # ---- freshness / stale-context detection
    def _current_hash(self, path: Path) -> str:
        st = path.stat()
        key = (st.st_mtime, st.st_size)
        cached = self._hash_cache.get(str(path))
        if not cached or cached[0] != key:
            cached = (key, _sha(path))
            self._hash_cache[str(path)] = cached
        return cached[1]

    def freshness(self, now: datetime | None = None) -> dict[str, tuple[str, str]]:
        """doc -> (status, reason). Status: fresh | stale | expired | superseded | modified | missing."""
        now, out = now or datetime.now(), {}
        for doc, rec in self.docs.items():
            path, meta = self.kb_dir / doc, rec["meta"]
            if not path.exists():
                out[doc] = ("missing", "The source file no longer exists.")
            elif self._current_hash(path) != rec["hash"]:
                out[doc] = ("modified", "The file changed after it was indexed — re-index to refresh.")
            elif meta.get("superseded_by"):
                out[doc] = ("superseded", f"Superseded by {meta['superseded_by']}.")
            elif (end := _date(meta.get("valid_until"))) and end < now:
                out[doc] = ("expired", f"Validity ended on {end:%Y-%m-%d}.")
            else:
                updated = _date(meta.get("last_updated")) or datetime.fromtimestamp(path.stat().st_mtime)
                age = (now - updated).days
                if age > self.stale_days:
                    out[doc] = ("stale", f"Last updated {updated:%Y-%m-%d} ({age} days ago).")
                else:
                    out[doc] = ("fresh", f"Last updated {updated:%Y-%m-%d}.")
        return out


## 4 · Hybrid retrieval + cross-encoder reranking
FAISS semantic search and BM25 keyword search are merged with weighted reciprocal-rank fusion; a cross-encoder then
re-scores the top candidates.

In [7]:
from __future__ import annotations

STOP = set("""a an and are as at be by can do does for from how i if in is it its me my of on or
our that the their there this to was we what when where which who why will with you your
many much long often must should tell please about""".split())


def _stem(w: str) -> str:
    if len(w) > 4 and w.endswith("ies"):
        return w[:-3] + "y"
    return w[:-1] if len(w) > 3 and w.endswith("s") and not w.endswith("ss") else w


def tokens(text: str) -> list[str]:
    return [_stem(w) for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP]


def relevance(reranker, query: str, texts: list[str]) -> np.ndarray:
    """Cross-encoder relevance as probabilities in 0..1 (sigmoid applied if raw logits come back)."""
    if not texts:
        return np.zeros(0)
    p = np.asarray(reranker.predict([(query, t) for t in texts]), dtype="float64")
    if p.size and (p.min() < 0 or p.max() > 1):
        p = 1 / (1 + np.exp(-np.clip(p, -30, 30)))
    return p


@dataclass
class Hit:
    chunk: Chunk
    semantic: float = 0.0          # cosine similarity
    sem_rank: int | None = None
    bm25_rank: int | None = None
    fused: float = 0.0             # weighted reciprocal-rank fusion, 0..1
    rerank: float | None = None    # cross-encoder probability, 0..1
    rank: int = 0

    @property
    def score(self) -> float:
        return self.rerank if self.rerank is not None else self.fused


class Retriever:
    def __init__(self, kb: KnowledgeBase, reranker=None):
        self.kb, self.reranker = kb, reranker
        self._bm25, self._bm25_ids, self._bm25_version = None, [], -1

    def _bm25_index(self) -> BM25Okapi | None:
        if self._bm25_version != self.kb.version:
            self._bm25_ids = sorted(self.kb.chunks)
            corpus = [tokens(f"{self.kb.chunks[i].title} {self.kb.chunks[i].heading or ''} {self.kb.chunks[i].text}")
                      for i in self._bm25_ids]
            self._bm25 = BM25Okapi(corpus) if corpus else None
            self._bm25_version = self.kb.version
        return self._bm25

    def search(self, query: str, pool: int = 20, semantic_weight: float = 0.6, rrf_k: int = 60,
               rerank: bool = True) -> list[Hit]:
        """Return the candidate pool, best first. Callers take ``[:top_k]`` for the final evidence."""
        kb = self.kb
        if not kb.chunks or not query.strip():
            return []
        breadth = max(pool * 2, 30)

        qvec = kb.embed_query(query)                                   # 1. semantic (FAISS)
        scores, ids = kb.index.search(qvec[None, :], min(breadth, len(kb.chunks)))
        sem = {int(i): (r + 1, float(s)) for r, (i, s) in enumerate(zip(ids[0], scores[0])) if i != -1}

        qtok, bm = tokens(query), {}                                   # 2. keyword (BM25)
        if qtok and self._bm25_index() is not None:
            raw = self._bm25.get_scores(qtok)
            top = sorted(((i, s) for i, s in zip(self._bm25_ids, raw) if s > 0), key=lambda x: -x[1])[:breadth]
            bm = {i: r + 1 for r, (i, _) in enumerate(top)}

        w, best = float(np.clip(semantic_weight, 0, 1)), 1 / (rrf_k + 1)   # 3. weighted RRF
        fused = {}
        for i in set(sem) | set(bm):
            s = w / (rrf_k + sem[i][0]) if i in sem else 0.0
            b = (1 - w) / (rrf_k + bm[i]) if i in bm else 0.0
            fused[i] = (s + b) / best
        top_ids = sorted(fused, key=lambda i: -fused[i])[:pool]

        hits = []
        for i in top_ids:
            c = kb.chunks[i]
            cos = sem[i][1] if i in sem else float(qvec @ kb.index.reconstruct(i))
            hits.append(Hit(c, cos, sem.get(i, (None,))[0], bm.get(i), fused[i]))

        if rerank and self.reranker is not None and hits:               # 4. cross-encoder rerank
            probs = relevance(self.reranker, query, [h.chunk.text for h in hits])
            for h, p in zip(hits, probs):
                h.rerank = float(p)
        hits.sort(key=lambda h: -h.score)
        for r, h in enumerate(hits, 1):
            h.rank = r
        return hits


## 5 · Grounded answers, evidence indicators, hallucination prevention
The answer is **extractive** — built only from sentences that exist in the retrieved passages. The system **abstains** when
evidence is weak, skips **superseded** passages, and warns about **stale** sources.

In [8]:
from __future__ import annotations

@dataclass
class Answer:
    text: str = ""
    citations: list[Hit] = field(default_factory=list)       # citations[n-1] is cited as [n]
    abstained: bool = False
    level: str = "none"                                       # strong | moderate | weak | none
    confidence: float = 0.0
    parts: dict = field(default_factory=dict)                 # relevance / coverage / freshness
    warnings: list[str] = field(default_factory=list)
    missing_terms: list[str] = field(default_factory=list)


def sentences(text: str) -> list[str]:
    out = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith(("#", "---")):
            continue
        line = re.sub(r"^([-*+]|\d+[.)])\s+", "", line)
        line = re.sub(r"\[([^\]]+)\]\([^)]*\)", r"\1", line)
        line = re.sub(r"[*_`]", "", line)
        out += [s.strip() for s in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9\"'(])", line) if len(s.strip()) > 25]
    return out


def _level(confidence: float) -> str:
    return "strong" if confidence >= 0.6 else "moderate" if confidence >= 0.38 else "weak" if confidence >= 0.2 else "none"


def answer(query: str, hits: list[Hit], kb: KnowledgeBase, reranker=None, top_k: int = 5,
           abstain_below: float = 0.10, max_sentences: int = 3) -> Answer:
    fresh = kb.freshness()
    top, warnings = hits[:top_k], []

    # skip passages whose document was superseded by another document that is in the index
    usable = []
    for h in top:
        newer = kb.docs.get(h.chunk.doc, {}).get("meta", {}).get("superseded_by")
        if newer and any(newer == d or newer == d.rsplit("/", 1)[-1] for d in kb.docs):
            warnings.append(f"Skipped “{h.chunk.title}” — superseded by {newer}.")
        else:
            usable.append(h)
    usable = usable or top
    if not usable:
        return Answer(abstained=True, warnings=["Nothing in the index matched the question."])

    # evidence assessment
    best = usable[0].score
    q_terms = set(tokens(query))
    evidence_terms = set(tokens(" ".join(f"{h.chunk.title} {h.chunk.heading or ''} {h.chunk.text}" for h in usable)))
    coverage = len(q_terms & evidence_terms) / len(q_terms) if q_terms else 1.0
    missing = sorted(q_terms - evidence_terms)
    if best < abstain_below or (coverage < 0.34 and usable[0].semantic < 0.55):
        why = (f"The best passage scored {best:.0%} relevance (threshold {abstain_below:.0%})."
               if best < abstain_below else "None of the question's key terms appear in the retrieved passages.")
        return Answer(abstained=True, parts={"relevance": best, "coverage": coverage}, missing_terms=missing,
                      warnings=warnings + [why])

    # choose the best verbatim sentences
    cands = [(h, i, s) for h in usable for i, s in enumerate(sentences(h.chunk.text))]
    if not cands:
        return Answer(abstained=True, warnings=warnings + ["No sentence in the retrieved passages answers the question."])
    if reranker is not None:
        sc = relevance(reranker, query, [s for _, _, s in cands])
    else:
        sc = np.array([len(q_terms & set(tokens(s))) / max(1, len(q_terms)) for _, _, s in cands])
    order = [int(k) for k in np.argsort(-sc)]
    floor = max(0.15, 0.6 * sc[order[0]])
    chosen = [k for k in order if sc[k] >= floor][:max_sentences]
    chosen.sort(key=lambda k: (cands[k][0].rank, cands[k][1]))

    citations: list[Hit] = []
    pieces: list[tuple[int, str]] = []
    for k in chosen:
        h = cands[k][0]
        if h not in citations:
            citations.append(h)
        n = citations.index(h) + 1
        if pieces and pieces[-1][0] == n:
            pieces[-1] = (n, pieces[-1][1] + " " + cands[k][2])
        else:
            pieces.append((n, cands[k][2]))
    text = " ".join(f"{t} [{n}]" for n, t in pieces)

    # freshness of what was actually cited
    ok = [fresh.get(h.chunk.doc, ("fresh", ""))[0] == "fresh" for h in citations]
    freshness = float(np.mean([1.0 if x else 0.3 for x in ok]))
    for n, h in enumerate(citations, 1):
        status, reason = fresh.get(h.chunk.doc, ("fresh", ""))
        if status != "fresh":
            warnings.append(f"[{n}] “{h.chunk.title}” is {status}: {reason}")
    if missing:
        warnings.append("Not found in the evidence: " + ", ".join(f"“{t}”" for t in missing[:5]) + ".")

    confidence = (0.6 * best + 0.4 * coverage) * (0.7 + 0.3 * freshness)
    return Answer(text, citations, False, _level(confidence), float(np.clip(confidence, 0, 1)),
                  {"relevance": best, "coverage": coverage, "freshness": freshness}, warnings, missing)


## 6 · Knowledge graph

In [9]:
# HTML/JS template for the interactive canvas graph (rendered inside an iframe below).
GRAPH_TEMPLATE = r'''<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<style>
html,body{margin:0;height:100%;background:transparent;color:#e6e9f5;font:12px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;overflow:hidden}
#root{position:relative;height:__HEIGHT__px;border-radius:16px;overflow:hidden;border:1px solid rgba(140,160,255,.2);
  background:radial-gradient(900px 480px at 15% 5%,rgba(99,102,241,.2),transparent 60%),radial-gradient(800px 480px at 95% 95%,rgba(34,211,238,.14),transparent 60%),#04050d}
canvas{position:absolute;inset:0;width:100%;height:100%;cursor:grab}
.glass{position:absolute;background:rgba(8,11,26,.7);backdrop-filter:blur(12px);border:1px solid rgba(140,160,255,.2);border-radius:10px;padding:7px 11px}
#legend{left:10px;bottom:10px;max-height:40%;overflow:auto}
.lg{display:flex;gap:8px;align-items:center;padding:2px 0;white-space:nowrap}
.lg i{width:9px;height:9px;border-radius:50%;box-shadow:0 0 8px currentColor}.lg b{font-weight:500;color:#fbbf24}
#buttons{right:10px;top:10px;display:flex;gap:6px;padding:4px}
button{background:none;border:0;color:#cbd2ee;cursor:pointer;font:inherit;padding:3px 9px;border-radius:6px}button:hover{background:rgba(255,255,255,.1)}
#tip{pointer-events:none;opacity:0;max-width:260px;transition:opacity .15s}#tip.on{opacity:1}#tip div{color:#8b93b8;margin-top:2px}
#empty{display:none;inset:0;align-items:center;justify-content:center;color:#8b93b8;position:absolute}
</style></head><body>
<div id="root"><canvas id="cv"></canvas>
<div id="legend" class="glass"></div>
<div id="buttons" class="glass"><button id="replay">↻ Replay</button><button id="fit">⤢ Fit</button></div>
<div id="tip" class="glass"></div><div id="empty">Index some documents to see the graph.</div></div>
<script>
(() => {
const D = __GRAPH__, root = document.getElementById('root'), cv = document.getElementById('cv'), ctx = cv.getContext('2d');
const COLORS = {stale:'#fb923c', expired:'#f87171', superseded:'#f472b6', modified:'#60a5fa', missing:'#94a3b8'};
const HL = D.highlight && D.highlight.evidence.length ? D.highlight : null;
const clamp = (v,a,b) => Math.max(a, Math.min(b, v)), ease = t => 1 - Math.pow(1 - t, 3);
const rgba = (h,a) => { const n = parseInt(h.slice(1),16); return `rgba(${n>>16&255},${n>>8&255},${n&255},${a})`; };
const nodes = D.nodes.map((n,i) => ({...n, i, x:n.x*340 + Math.random()*6, y:n.y*340 + Math.random()*6, vx:0, vy:0, fix:false, ev:null, cons:false, lit:0,
  r:n.kind === 'doc' ? 11 : 4, q:n.kind === 'doc' ? 5 : 1}));
const adj = nodes.map(() => []); D.edges.forEach(e => { adj[e.s].push(e); adj[e.t].push(e); });
nodes.forEach(n => { if (n.kind === 'chunk') n.r += Math.min(3, Math.sqrt(adj[n.i].filter(e => e.kind === 'sim').length)); });
const evNodes = []; if (HL) { HL.evidence.forEach(e => { nodes[e.i].ev = e; evNodes.push(nodes[e.i]); }); HL.considered.forEach(i => nodes[i].cons = true); }
const col = n => D.docs[n.doc].color;
let W, H, dpr = 1, k = 1, tx = 0, ty = 0, alpha = 1, hover = -1, sel = -1, t0 = performance.now(), moved = false, q = null;
const glow = {}; const sprite = c => glow[c] || (glow[c] = (() => { const s = document.createElement('canvas'); s.width = s.height = 96; const g = s.getContext('2d'), r = g.createRadialGradient(48,48,0,48,48,48);
  r.addColorStop(0, rgba(c,.9)); r.addColorStop(.25, rgba(c,.4)); r.addColorStop(1, rgba(c,0)); g.fillStyle = r; g.fillRect(0,0,96,96); return s; })());

function fit() { if (!nodes.length) return; const xs = nodes.map(n => n.x).concat(q ? [q.x] : []), ys = nodes.map(n => n.y).concat(q ? [q.y] : []);
  const x0 = Math.min(...xs), x1 = Math.max(...xs), y0 = Math.min(...ys), y1 = Math.max(...ys);
  k = clamp(Math.min(W / (x1-x0+200), H / (y1-y0+200)), .15, 2.5); tx = W/2 - (x0+x1)/2*k; ty = H/2 - (y0+y1)/2*k; }
function resize() { const r = root.getBoundingClientRect(); W = r.width; H = r.height; dpr = Math.min(devicePixelRatio || 1, 2); cv.width = W*dpr; cv.height = H*dpr; if (!moved) fit(); }
new ResizeObserver(resize).observe(root);

function physics() { if (alpha < .01) return; const n = nodes.length;
  for (let i = 0; i < n; i++) for (let j = i+1; j < n; j++) { const a = nodes[i], b = nodes[j]; let dx = b.x-a.x, dy = b.y-a.y, d2 = dx*dx+dy*dy;
    if (d2 > 90000) continue; if (d2 < 9) { dx = Math.random()-.5; dy = Math.random()-.5; d2 = 9; } const d = Math.sqrt(d2), f = a.q*b.q*480/d2*alpha;
    a.vx -= dx/d*f; a.vy -= dy/d*f; b.vx += dx/d*f; b.vy += dy/d*f; }
  for (const e of D.edges) { const a = nodes[e.s], b = nodes[e.t], dx = b.x-a.x, dy = b.y-a.y, d = Math.hypot(dx,dy) || 1;
    const rest = e.kind === 'doc' ? 46 : 110 - 60*e.w, str = e.kind === 'doc' ? .06 : .015 + .02*e.w, f = (d-rest)*str*alpha;
    a.vx += dx/d*f; a.vy += dy/d*f; b.vx -= dx/d*f; b.vy -= dy/d*f; }
  for (const a of nodes) { if (a.fix) continue; a.vx -= a.x*.0035*alpha; a.vy -= a.y*.0035*alpha; a.vx *= .78; a.vy *= .78; a.x += a.vx; a.y += a.vy; }
  alpha *= .992; }

const dim = n => HL ? (n.ev ? 1 : n.cons ? .5 : n.kind === 'doc' ? .6 : .2) : 1;
function draw(now) {
  const tt = now/1000, since = now - t0; ctx.setTransform(dpr,0,0,dpr,0,0); ctx.clearRect(0,0,W,H); ctx.setTransform(dpr*k,0,0,dpr*k,dpr*tx,dpr*ty);
  if (HL) { evNodes.forEach(n => n.lit = ease(clamp((since - 300 - (n.ev.rank-1)*150)/600, 0, 1)));
    const cx = evNodes.reduce((s,n) => s+n.x, 0)/evNodes.length, cy = evNodes.reduce((s,n) => s+n.y, 0)/evNodes.length;
    const gx = nodes.reduce((s,n) => s+n.x, 0)/nodes.length, gy = nodes.reduce((s,n) => s+n.y, 0)/nodes.length;
    let dx = cx-gx, dy = cy-gy, dl = Math.hypot(dx,dy); if (dl < 1) { dx = 0; dy = -1; dl = 1; }
    const tx2 = cx + dx/dl*150, ty2 = cy + dy/dl*150; q = q ? {x:q.x + (tx2-q.x)*.06, y:q.y + (ty2-q.y)*.06} : {x:tx2, y:ty2}; }
  ctx.lineCap = 'round'; ctx.lineWidth = 1/k;
  for (const e of D.edges) { const a = nodes[e.s], b = nodes[e.t], m = Math.min(dim(a), dim(b)) > .3 ? 1 : .4;
    ctx.strokeStyle = e.kind === 'doc' ? `rgba(148,163,255,${.1*m})` : `rgba(103,232,249,${(.15 + .4*e.w)*m})`;
    ctx.beginPath(); ctx.moveTo(a.x,a.y); ctx.lineTo(b.x,b.y); ctx.stroke();
    if (e.kind === 'sim' && a.ev && b.ev) { ctx.save(); ctx.shadowColor = '#22d3ee'; ctx.shadowBlur = 14; ctx.strokeStyle = `rgba(34,211,238,${.95*Math.min(a.lit,b.lit)})`; ctx.lineWidth = 2/k; ctx.stroke(); ctx.restore(); } }
  const f = hover >= 0 ? hover : sel; if (f >= 0) for (const e of adj[f]) { ctx.save(); ctx.shadowColor = '#f0abfc'; ctx.shadowBlur = 10; ctx.strokeStyle = 'rgba(240,171,252,.85)'; ctx.lineWidth = 1.6/k;
    ctx.beginPath(); ctx.moveTo(nodes[e.s].x, nodes[e.s].y); ctx.lineTo(nodes[e.t].x, nodes[e.t].y); ctx.stroke(); ctx.restore(); }
  ctx.globalCompositeOperation = 'lighter';
  for (const n of nodes) { const s = n.ev ? (n.r + 6 + 14*n.ev.score)*(1 + .6*n.lit) : n.r*(n.kind === 'doc' ? 4 : 5); ctx.globalAlpha = dim(n)*(n.ev ? .4 + .6*n.lit : .7); ctx.drawImage(sprite(col(n)), n.x-s, n.y-s, s*2, s*2); }
  ctx.globalCompositeOperation = 'source-over';
  if (HL && q) evNodes.forEach((n, idx) => { const p = ease(clamp((since - 200 - (n.ev.rank-1)*150)/700, 0, 1)); if (p <= 0) return;
    const mx = (q.x+n.x)/2, my = (q.y+n.y)/2, dx = n.x-q.x, dy = n.y-q.y, len = Math.hypot(dx,dy) || 1, off = len*.2*(idx % 2 ? 1 : -1), c = [mx - dy/len*off, my + dx/len*off];
    const at = u => [(1-u)*(1-u)*q.x + 2*(1-u)*u*c[0] + u*u*n.x, (1-u)*(1-u)*q.y + 2*(1-u)*u*c[1] + u*u*n.y];
    ctx.save(); ctx.shadowColor = '#22d3ee'; ctx.shadowBlur = 12; ctx.strokeStyle = '#67e8f9'; ctx.globalAlpha = .7; ctx.lineWidth = (1 + 2*n.ev.score)/k; ctx.beginPath();
    for (let s = 0; s <= 24; s++) { const pt = at(p*s/24); s ? ctx.lineTo(...pt) : ctx.moveTo(...pt); } ctx.stroke(); ctx.restore();
    if (p >= 1) { ctx.globalCompositeOperation = 'lighter'; for (let m = 0; m < 3; m++) { const ph = (now/1500 + m/3 + n.ev.rank*.17) % 1, pt = at(ph), s = 8/k; ctx.globalAlpha = Math.sin(ph*Math.PI); ctx.drawImage(sprite('#67e8f9'), pt[0]-s, pt[1]-s, s*2, s*2); } ctx.globalCompositeOperation = 'source-over'; } });
  for (const n of nodes) { ctx.globalAlpha = dim(n); const c = col(n);
    if (n.kind === 'doc') { ctx.beginPath(); ctx.arc(n.x, n.y, n.r*.62, 0, 7); ctx.fillStyle = '#04050d'; ctx.fill(); ctx.lineWidth = 2/k; ctx.strokeStyle = c; ctx.stroke();
      ctx.beginPath(); ctx.arc(n.x, n.y, n.r*.22, 0, 7); ctx.fillStyle = c; ctx.fill(); }
    else { const r = n.ev ? (5 + 8*n.ev.score)*(.55 + .45*n.lit) : n.r;
      if (n.ev) { ctx.beginPath(); ctx.arc(n.x, n.y, r*(1.5 + .1*Math.sin(tt*3)), 0, 7); ctx.strokeStyle = c; ctx.lineWidth = 1.3/k; ctx.globalAlpha = .5*n.lit; ctx.stroke(); ctx.globalAlpha = 1; }
      ctx.beginPath(); ctx.arc(n.x, n.y, r, 0, 7); ctx.fillStyle = c; ctx.fill();
      if (n.ev) { ctx.beginPath(); ctx.arc(n.x, n.y, r*.45, 0, 7); ctx.fillStyle = '#fff'; ctx.fill(); } }
    if (COLORS[n.status]) { ctx.save(); ctx.setLineDash([3/k, 3/k]); ctx.lineDashOffset = -tt*6/k; ctx.beginPath(); ctx.arc(n.x, n.y, n.r + 6, 0, 7); ctx.strokeStyle = COLORS[n.status]; ctx.lineWidth = 1.2/k; ctx.stroke(); ctx.restore(); }
    if (n.i === hover || n.i === sel) { ctx.beginPath(); ctx.arc(n.x, n.y, n.r + 8, 0, 7); ctx.strokeStyle = n.i === sel ? '#22d3ee' : '#fff'; ctx.lineWidth = 1.5/k; ctx.stroke(); } }
  ctx.globalAlpha = 1; ctx.textAlign = 'center'; ctx.textBaseline = 'middle';
  const text = (s, x, y, px, c) => { ctx.font = `600 ${px/k}px system-ui,sans-serif`; ctx.lineWidth = 3/k; ctx.strokeStyle = 'rgba(4,5,13,.9)'; ctx.strokeText(s, x, y); ctx.fillStyle = c; ctx.fillText(s, x, y); };
  for (const n of nodes) { if (n.kind === 'doc') { ctx.globalAlpha = Math.min(1, dim(n) + .2); text(n.title, n.x, n.y + n.r + 11/k, 11, '#e6e9f5'); }
    else if (n.ev) { const r = (5 + 8*n.ev.score)*(.55 + .45*n.lit); ctx.globalAlpha = n.lit; ctx.beginPath(); ctx.arc(n.x + r, n.y - r, 6.5/k, 0, 7); ctx.fillStyle = '#04121a'; ctx.fill(); ctx.strokeStyle = '#22d3ee'; ctx.lineWidth = 1.2/k; ctx.stroke();
      text(String(n.ev.rank), n.x + r, n.y - r, 9, '#ecfeff'); text(n.title + ' · #' + n.n, n.x, n.y + r + 11/k, 10.5, '#ecfeff'); } }
  if (HL && q) { ctx.globalAlpha = 1; ctx.globalCompositeOperation = 'lighter'; const s = 40/k*(1 + .1*Math.sin(tt*3)); ctx.drawImage(sprite('#67e8f9'), q.x-s, q.y-s, s*2, s*2); ctx.globalCompositeOperation = 'source-over';
    const rp = (since/2600) % 1; ctx.beginPath(); ctx.arc(q.x, q.y, (10 + rp*80)/k, 0, 7); ctx.strokeStyle = '#67e8f9'; ctx.globalAlpha = (1-rp)*.45; ctx.lineWidth = 1.3/k; ctx.stroke(); ctx.globalAlpha = 1;
    ctx.beginPath(); ctx.arc(q.x, q.y, 3.5/k, 0, 7); ctx.fillStyle = '#fff'; ctx.fill(); text('QUERY', q.x, q.y - 15/k, 10, '#67e8f9'); }
  ctx.globalAlpha = 1; }
(function loop(now) { requestAnimationFrame(loop); physics(); draw(now); })(performance.now());

const tip = document.getElementById('tip');
const world = (x, y) => [(x - tx)/k, (y - ty)/k];
function pick(x, y) { const [wx, wy] = world(x, y); let b = -1, bd = 1e9; for (const n of nodes) { const d = Math.hypot(n.x-wx, n.y-wy); if (d < Math.max(n.r + 4, 10/k) && d < bd && dim(n) > .15) { bd = d; b = n.i; } } return b; }
let drag = null;
cv.onpointerdown = e => { const r = cv.getBoundingClientRect(), x = e.clientX - r.left, y = e.clientY - r.top, h = pick(x, y); cv.setPointerCapture(e.pointerId); drag = {x, y, h, tx, ty, m:false}; if (h >= 0) nodes[h].fix = true; };
cv.onpointermove = e => { const r = cv.getBoundingClientRect(), x = e.clientX - r.left, y = e.clientY - r.top;
  if (drag) { if (Math.hypot(x-drag.x, y-drag.y) > 3) drag.m = true; if (drag.m) { if (drag.h >= 0) { [nodes[drag.h].x, nodes[drag.h].y] = world(x, y); alpha = Math.max(alpha, .25); } else { tx = drag.tx + x - drag.x; ty = drag.ty + y - drag.y; moved = true; } } return; }
  hover = pick(x, y); cv.style.cursor = hover >= 0 ? 'pointer' : 'grab';
  if (hover >= 0) { const n = nodes[hover]; tip.replaceChildren(Object.assign(document.createElement('b'), {textContent:n.title + (n.kind === 'chunk' ? ' · chunk ' + n.n : '')}), Object.assign(document.createElement('div'), {textContent:n.snip}));
    tip.style.left = Math.min(W - 280, x + 14) + 'px'; tip.style.top = Math.min(H - 100, y + 14) + 'px'; tip.className = 'glass on'; } else tip.className = 'glass'; };
cv.onpointerup = () => { if (!drag) return; if (drag.h >= 0) nodes[drag.h].fix = false; if (!drag.m) sel = drag.h === sel ? -1 : drag.h; drag = null; };
cv.onpointerleave = () => { hover = -1; tip.className = 'glass'; };
cv.onwheel = e => { e.preventDefault(); moved = true; const r = cv.getBoundingClientRect(), x = e.clientX - r.left, y = e.clientY - r.top, [wx, wy] = world(x, y); k = clamp(k*Math.exp(-e.deltaY*.0015), .1, 6); tx = x - wx*k; ty = y - wy*k; };
cv.ondblclick = () => { moved = false; fit(); };
document.getElementById('fit').onclick = () => { moved = false; fit(); };
document.getElementById('replay').onclick = () => { t0 = performance.now(); };
if (!HL) document.getElementById('replay').style.display = 'none';

const lg = document.getElementById('legend');
D.docs.forEach(d => { const r = document.createElement('div'); r.className = 'lg'; const i = document.createElement('i'); i.style.background = i.style.color = d.color; r.append(i, document.createTextNode(d.title));
  if (d.status !== 'fresh') { const w = document.createElement('b'); w.textContent = '⚠ ' + d.status; w.title = d.reason; r.append(w); } lg.append(r); });
if (!nodes.length) { document.getElementById('empty').style.display = 'flex'; lg.style.display = 'none'; }
resize();
})();
</script></body></html>
'''


In [10]:
from __future__ import annotations

PALETTE = ["#22d3ee", "#a78bfa", "#f472b6", "#34d399", "#fbbf24", "#60a5fa", "#fb7185", "#2dd4bf"]


def build_graph(kb: KnowledgeBase, hits: list[Hit] | None = None, cited: dict[int, int] | None = None,
                top_k: int = 5, neighbors: int = 3, threshold: float = 0.45) -> dict:
    """Nodes + edges + the retrieval highlight, ready for the canvas renderer."""
    ids, V = kb.matrix()
    if not ids:
        return {"nodes": [], "edges": [], "docs": [], "highlight": None}
    pos = {cid: j for j, cid in enumerate(ids)}
    fresh = kb.freshness()

    # seed the layout with a 2-D PCA of the embeddings; the browser force simulation refines it
    X = V - V.mean(axis=0)
    xy = X @ np.linalg.svd(X, full_matrices=False)[2][:2].T if len(V) > 2 else np.random.rand(len(V), 2)
    xy = xy / (np.abs(xy).max() or 1)

    doc_ids = sorted({kb.chunks[i].doc for i in ids})
    docs = [{"title": kb.docs[d]["title"], "color": PALETTE[k % len(PALETTE)], "status": fresh[d][0],
             "reason": fresh[d][1]} for k, d in enumerate(doc_ids)]
    doc_no = {d: k for k, d in enumerate(doc_ids)}

    nodes = []
    for j, cid in enumerate(ids):
        c = kb.chunks[cid]
        nodes.append({"kind": "chunk", "doc": doc_no[c.doc], "title": c.title, "n": c.n + 1,
                      "snip": " ".join(c.text.split())[:200], "status": fresh[c.doc][0],
                      "x": round(float(xy[j, 0]), 4), "y": round(float(xy[j, 1]), 4)})
    hub = {}
    for d in doc_ids:
        members = [pos[i] for i in ids if kb.chunks[i].doc == d]
        hub[d] = len(nodes)
        nodes.append({"kind": "doc", "doc": doc_no[d], "title": kb.docs[d]["title"], "status": fresh[d][0],
                      "snip": f"{len(members)} chunks · {fresh[d][0]}",
                      "x": round(float(xy[members, 0].mean()), 4), "y": round(float(xy[members, 1].mean()), 4)})

    edges, seen = [], set()

    def link(a: int, b: int, kind: str, w: float = 1.0) -> None:
        key = (min(a, b), max(a, b), kind)
        if a != b and key not in seen:
            seen.add(key)
            edges.append({"s": a, "t": b, "kind": kind, "w": round(w, 3)})

    for j, cid in enumerate(ids):
        link(j, hub[kb.chunks[cid].doc], "doc")
    if len(ids) > 1:
        S = V @ V.T
        np.fill_diagonal(S, -1)
        k = min(neighbors, len(ids) - 1)
        for a in range(len(ids)):
            for b in np.argpartition(-S[a], k - 1)[:k]:
                if S[a, b] >= threshold:
                    link(a, int(b), "sim", float(S[a, b]))

    highlight = None
    if hits:
        shown = hits[:top_k]
        highlight = {
            "evidence": [{"i": pos[h.chunk.id], "rank": h.rank, "score": round(h.score, 3),
                          "cite": (cited or {}).get(h.chunk.id)} for h in shown],
            "considered": [pos[h.chunk.id] for h in hits[top_k:]],
        }
    return {"nodes": nodes, "edges": edges, "docs": docs, "highlight": highlight}


def render_graph(graph: dict, height: int = 620) -> str:
    data = json.dumps(graph, ensure_ascii=False).replace("<", "\\u003c").replace(">", "\\u003e").replace("&", "\\u0026")
    return GRAPH_TEMPLATE.replace("__GRAPH__", data).replace("__HEIGHT__", str(height))


## 7 · Sample documents
Creates a small `knowledge_base/` folder the first time (skipped if it already has files). It includes one **superseded** policy and one **expired** handbook so you can see the stale-context warnings.

In [11]:
KB_DIR, INDEX_DIR = Path('knowledge_base'), Path('.memory_lane')

SAMPLES = {
    "engineering_onboarding.md": """---
title: Engineering Onboarding Guide
last_updated: 2026-08-20
---

# Engineering Onboarding Guide

Welcome to Aurora Labs engineering. This guide covers your first 90 days.

## Week one

On day one you receive your laptop and an onboarding buddy. By the end of the first week you should have your development environment running, merged a small documentation fix, and met every member of your team. Access to repositories, the CI system and the staging cluster is requested through the IT portal; approvals normally take one business day.

## Development environment

We standardise on Python 3.12 managed with uv, Node 22 for front-end work, and Docker for local services. Clone the monorepo, run `make bootstrap`, and then `make test` to confirm everything passes before you change anything. If bootstrap fails, ask in the engineering-help channel rather than working around it.

## Code review

Every change needs two approvals before it can merge, and at least one reviewer must come from outside your immediate team. Keep pull requests under 400 lines where possible. Reviewers aim to respond within one business day. Never merge your own change without the required approvals, even for urgent fixes; use the emergency-merge process instead.

## On-call

You shadow the on-call engineer starting in week four and join the rotation after your first 90 days. Before joining the rotation, read the incident response runbook and complete the pager training session.

## 30, 60 and 90 day goals

By day 30 you ship your first feature behind a flag. By day 60 you own a small service or component. By day 90 you lead a design review and join the on-call rotation.
""",

    "incident_response_runbook.md": """---
title: Incident Response Runbook
last_updated: 2026-07-02
---

# Incident Response Runbook

This runbook describes how Aurora Labs detects, handles and learns from production incidents.

## Severity levels

SEV1 means a full outage or data loss affecting many customers. The on-call engineer must acknowledge a SEV1 page within 5 minutes and an incident commander is appointed immediately. SEV2 is a major feature outage or severe degradation, acknowledged within 15 minutes. SEV3 is a minor degradation with a workaround, handled during business hours. SEV4 is a cosmetic issue or a request that is tracked as a normal ticket.

## Roles

Every SEV1 and SEV2 incident has three roles. The incident commander coordinates the response and makes decisions. The communications lead posts updates to the status page and to affected customers. The scribe records a timeline of actions and decisions in the incident channel.

## Communication cadence

For a SEV1 the status page is updated every 30 minutes until the incident is resolved, even if there is no new information. For a SEV2 updates go out every hour. Internal updates are posted in the incident channel as events happen.

## Resolution and follow-up

An incident is resolved only when customer impact has ended and monitoring has been stable for 30 minutes. A blameless postmortem must be written within 5 business days of any SEV1 or SEV2. The postmortem lists contributing factors, what went well, and action items with named owners and due dates.

## Escalation

If the on-call engineer does not acknowledge a page within the target time, it automatically escalates to the secondary on-call and then to the engineering manager.
""",

    "rag_primer.md": """---
title: RAG Primer
last_updated: 2026-10-01
---

# Retrieval-Augmented Generation Primer

Retrieval-augmented generation, or RAG, answers questions by first retrieving relevant passages from a knowledge base and then composing an answer grounded in those passages.

## Chunking

Documents are split into chunks so that each passage is small enough to embed and to read. Recursive chunking tries large separators first, such as section headings and paragraphs, and falls back to sentences and words only when a piece is still too long. A small overlap between neighbouring chunks keeps sentences that straddle a boundary retrievable.

## Embeddings and vector search

An embedding model maps text to a vector so that passages with similar meaning end up close together. With normalised vectors, cosine similarity equals the inner product. FAISS is a library for fast similarity search over such vectors; a flat inner-product index gives exact results and is ideal for collections up to a few hundred thousand chunks.

## Keyword search with BM25

BM25 scores passages by how often the query terms appear, weighted by how rare each term is across the collection. It is excellent at exact names, identifiers and numbers, where embeddings can be vague.

## Hybrid retrieval and reciprocal rank fusion

Dense and keyword search fail in different ways, so strong systems run both. Reciprocal rank fusion merges the two ranked lists without needing comparable scores: each passage earns 1 divided by the sum of a constant k and its rank in each list, and the constant k is usually 60.

## Reranking

A cross-encoder reads the query and a passage together and outputs a relevance score. It is far more accurate than comparing precomputed vectors but too slow to run on everything, so it is applied only to the top candidates from hybrid retrieval.

## Preventing hallucinations

Ground every claim in a cited passage. Abstain when the best evidence is weak. Verify generated sentences against the sources and remove those that are unsupported. Flag evidence that is stale, expired or superseded so users do not trust outdated answers.
""",

    "remote_work_policy.md": """---
title: Remote Work Policy
last_updated: 2026-09-12
---

# Remote Work Policy

Aurora Labs supports flexible work. This policy explains where, when and how employees may work outside the office.

## Eligibility and schedule

Employees may work remotely up to 3 days per week. The remaining days are spent in the office so teams can collaborate in person. Everyone is expected to be reachable during core hours, which are 10:00 to 15:00 in their local time zone. Managers may agree on a different pattern for a team, but the 3-day limit applies unless a VP approves a fully remote arrangement.

## Fully remote roles

A fully remote arrangement needs written approval from a VP and must be reviewed every 12 months. Fully remote employees travel to the nearest hub for the quarterly team week, and the company covers that travel.

## Equipment and expenses

New remote employees receive a one-time home-office stipend of $750 for a desk, chair or monitor. Aurora Labs also reimburses $50 per month toward home internet. Receipts must be submitted through the expense portal within 60 days.

## Working from abroad

Employees may work from another country for up to 20 days per calendar year with manager approval. Requests must be filed at least 30 days in advance because tax and legal teams need to review them. Work from abroad is not allowed in countries where Aurora Labs has no legal entity and no approved employer-of-record partner.

## Security while remote

Use the company VPN whenever you are on public Wi-Fi. Never work on customer data from a shared or unlocked device. Lost or stolen laptops must be reported to the security team immediately.
""",

    "remote_work_policy_2022.md": """---
title: Remote Work Policy (2022 edition)
last_updated: 2022-01-15
superseded_by: remote_work_policy.md
---

# Remote Work Policy (2022 edition)

This is the original hybrid-work policy adopted in January 2022.

## Schedule

Employees may work remotely up to 2 days per week. The other three days must be spent in the office. Core hours are 9:30 to 16:00 in the employee's local time zone.

## Equipment

Remote employees receive a one-time home-office stipend of $300. Internet costs are not reimbursed.

## Travel and location

Employees must work from the country where they are employed. Working from another country is not permitted under any circumstances.

## Review

This policy will be reviewed after the first year of hybrid work.
""",

    "security_handbook.md": """---
title: Security Handbook
last_updated: 2025-05-30
valid_until: 2026-06-30
---

# Security Handbook

This handbook sets the baseline security rules for everyone at Aurora Labs. It is reviewed annually and is valid until the end of June 2026.

## Passwords and authentication

Passwords must be at least 14 characters long. Use the company password manager and never reuse a password across services. Multi-factor authentication is required for email, the VPN, source control and every production system. Hardware security keys are required for administrators.

## Devices

All laptops must use full-disk encryption and install operating system updates within 7 days of release. If a device is lost or stolen, report it to the security team within 1 hour so it can be remotely wiped.

## Phishing

Report suspicious messages with the Report Phishing button in your mail client. Do not click links or open attachments from unknown senders. The security team runs simulated phishing campaigns every quarter and uses them for training, not for discipline.

## Data classification

Data is classified as Public, Internal, Confidential or Restricted. Customer data is Restricted and may only be accessed through approved tooling, never copied to personal devices or shared in chat.

## Vendors

New vendors that handle Confidential or Restricted data need a security review before a contract is signed. Reviews take about two weeks.
""",

}

KB_DIR.mkdir(exist_ok=True)
if not any(KB_DIR.iterdir()):
    for name, text in SAMPLES.items():
        (KB_DIR / name).write_text(text, encoding='utf-8')
    print(f'Created {len(SAMPLES)} sample documents in {KB_DIR.resolve()}')
else:
    print(f'Using existing documents in {KB_DIR.resolve()}')


Using existing documents in C:\Users\Swagat\Downloads\memory lane\memory-lane\knowledge_base


## 8 · Load models & build the index
Change `EMBEDDING_MODEL` to try `BAAI/bge-small-en-v1.5` (the index rebuilds automatically).

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL, encode_kwargs={"normalize_embeddings": True})
reranker = CrossEncoder(RERANKER_MODEL)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (11 outputs left out)

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [13]:
kb = KnowledgeBase(KB_DIR, INDEX_DIR, embeddings, EMBEDDING_MODEL)
retriever = Retriever(kb, reranker)
report = kb.sync()
print("Sync:", {k: v for k, v in report.items() if v} or "index already up to date")
print(f"{len(kb.docs)} documents · {len(kb.chunks)} chunks")


Sync: {'added': ['engineering_onboarding.md', 'incident_response_runbook.md', 'rag_primer.md', 'remote_work_policy.md', 'remote_work_policy_2022.md', 'security_handbook.md']}
6 documents · 18 chunks


## 9 · The app
Type a question and press **Ask**, or call `ask("your question")` directly. **Sync** re-indexes changed files;
**Rebuild** re-embeds everything.

In [14]:
LEVEL_ICON = {"strong": "🟢", "moderate": "🟡", "weak": "🟠", "none": "🔴"}
STATUS_ICON = {"fresh": "🟢", "stale": "🟠", "expired": "🔴", "superseded": "🟣", "modified": "🔵", "missing": "⚫"}
e = html.escape


def _card(inner, border="rgba(34,211,238,.35)", bg="rgba(34,211,238,.06)"):
    return (f'<div style="border:1px solid {border};background:{bg};border-radius:12px;padding:12px 16px;'
            f'margin:8px 0;font-family:system-ui,sans-serif;line-height:1.5">{inner}</div>')


def show_graph(graph, height=620):
    doc = render_graph(graph, height)
    frame = (f'<iframe srcdoc="{html.escape(doc, quote=True)}" '
             f'style="width:100%;height:{height + 16}px;border:0;border-radius:14px"></iframe>')
    display({"text/html": frame}, raw=True)


def documents_table():
    fresh = kb.freshness()
    return pd.DataFrame([{
        "document": rec["title"], "file": doc, "chunks": len(rec["chunk_ids"]),
        "status": f"{STATUS_ICON[fresh[doc][0]]} {fresh[doc][0]}", "detail": fresh[doc][1],
    } for doc, rec in kb.docs.items()])


def ask(question, top_k=5, semantic_weight=0.6, use_rerank=True, show_details=True):
    """Retrieve, answer with citations, and draw the evidence graph. Returns the Answer object."""
    if not kb.chunks:
        display(HTML(_card("The index is empty — add documents to <code>knowledge_base/</code> and press Sync.", "#f59e0b", "rgba(245,158,11,.08)")))
        return None
    hits = retriever.search(question, semantic_weight=semantic_weight, rerank=use_rerank)
    res = answer(question, hits, kb, reranker if use_rerank else None, top_k=top_k)

    if res.abstained:
        display(HTML(_card("I couldn't find enough evidence in your documents to answer that reliably.", "#f59e0b", "rgba(245,158,11,.08)")))
    else:
        tiles = "".join(
            f'<div style="flex:1;min-width:110px"><div style="font-size:11px;opacity:.65">{k}</div>'
            f'<div style="font-size:22px;font-weight:600">{v}</div></div>'
            for k, v in [("Evidence", f"{LEVEL_ICON[res.level]} {res.level}"), ("Confidence", f"{res.confidence:.0%}"),
                         ("Relevance", f"{res.parts['relevance']:.0%}"), ("Term coverage", f"{res.parts['coverage']:.0%}"),
                         ("Freshness", f"{res.parts['freshness']:.0%}")])
        display(HTML(_card(f'<div style="font-size:16px">{e(res.text)}</div>')
                     + f'<div style="display:flex;gap:12px;flex-wrap:wrap;margin:6px 4px">{tiles}</div>'))
    for w in res.warnings:
        display(HTML(_card("⚠️ " + e(w), "#f59e0b", "rgba(245,158,11,.08)")))
    for n, h in enumerate(res.citations, 1):
        where = e(h.chunk.title) + (f" › {e(h.chunk.heading)}" if h.chunk.heading else "") + (f" · p.{h.chunk.page}" if h.chunk.page else "")
        display(HTML(f'<details style="margin:4px 0;font-family:system-ui,sans-serif"><summary><b>[{n}]</b> {where}</summary>'
                     f'<div style="padding:8px 12px;opacity:.85">{e(h.chunk.text)}<br><small>{e(h.chunk.doc)} · relevance {h.score:.0%} · semantic {h.semantic:.2f}</small></div></details>'))

    if show_details:
        used = {c.chunk.id for c in res.citations}
        display(pd.DataFrame([{
            "rank": h.rank, "document": h.chunk.title, "section": h.chunk.heading, "rerank": h.rerank,
            "fused": round(h.fused, 3), "semantic": round(h.semantic, 3), "semantic rank": h.sem_rank,
            "keyword rank": h.bm25_rank, "used in answer": h.chunk.id in used,
        } for h in hits]).style.hide(axis="index").format({"rerank": "{:.2f}"}, na_rep="–"))
    cited = {h.chunk.id: n for n, h in enumerate(res.citations, 1)}
    show_graph(build_graph(kb, hits, cited, top_k))
    return res


# ---- controls
question = widgets.Text(placeholder="e.g. How many days a week can I work remotely?", layout=widgets.Layout(width="70%"))
ask_btn = widgets.Button(description="Ask", button_style="primary", icon="search")
sync_btn = widgets.Button(description="Sync", icon="refresh", tooltip="Index new files, re-index changed ones, drop deleted ones")
rebuild_btn = widgets.Button(description="Rebuild", icon="database", tooltip="Re-embed everything from scratch")
top_k_w = widgets.IntSlider(value=5, min=1, max=10, description="Evidence chunks", style={"description_width": "110px"})
weight_w = widgets.FloatSlider(value=0.6, min=0, max=1, step=0.05, description="Semantic ⟷ keyword", style={"description_width": "130px"})
rerank_w = widgets.Checkbox(value=True, description="Cross-encoder reranking")
out = widgets.Output()


def _run(_=None):
    with out:
        clear_output(wait=True)
        if question.value.strip():
            ask(question.value, top_k_w.value, weight_w.value, rerank_w.value)


def _sync(rebuild):
    with out:
        clear_output(wait=True)
        rep = kb.sync(rebuild=rebuild)
        print({k: v for k, v in rep.items() if v} or "Index is up to date.", f"· {len(kb.docs)} documents · {len(kb.chunks)} chunks")
        display(documents_table().style.hide(axis="index"))
        show_graph(build_graph(kb))


ask_btn.on_click(_run)
question.continuous_update = False
question.observe(lambda change: _run(), names='value')
sync_btn.on_click(lambda _: _sync(False))
rebuild_btn.on_click(lambda _: _sync(True))

display(widgets.VBox([
    widgets.HBox([question, ask_btn, sync_btn, rebuild_btn]),
    widgets.HBox([top_k_w, weight_w, rerank_w]),
    out,
]))


## 10 · Try it directly (optional)
`ask(...)` works without the widgets — handy for scripting. Good questions for the sample documents:

- *How many days per week can I work remotely?* → cites the current policy and **skips the superseded 2022 edition**
- *How long must passwords be?* → answers, but warns the **Security Handbook has expired**
- *How quickly must a SEV1 page be acknowledged?*
- *What is reciprocal rank fusion?*
- *What is the airspeed velocity of an unladen swallow?* → **abstains** instead of guessing

In [ ]:
res = ask("How many remote days are allowed?")

rank,document,section,rerank,fused,semantic,semantic rank,keyword rank,used in answer
1,Remote Work Policy (2022 edition),Remote Work Policy (2022 edition),0.00,0.987000,0.238000,1,3.000000,False
2,Remote Work Policy,Remote Work Policy,0.00,0.919000,0.061000,8,4.000000,False
3,Security Handbook,Data classification,0.00,0.930000,0.106000,6,5.000000,False
4,RAG Primer,Preventing hallucinations,0.00,0.563000,0.121000,5,–,False
5,Engineering Onboarding Guide,Code review,0.00,0.572000,0.134000,4,–,False
6,RAG Primer,Retrieval-Augmented Generation Primer,0.00,0.888000,0.009000,14,2.000000,False
7,Incident Response Runbook,Roles,0.00,0.960000,0.159000,2,6.000000,False
8,Engineering Onboarding Guide,Engineering Onboarding Guide,0.00,0.475000,-0.042000,17,–,False
9,RAG Primer,Hybrid retrieval and reciprocal rank fusion,0.00,0.482000,-0.036000,16,–,False
10,Security Handbook,Security Handbook,0.00,0.508000,0.022000,12,–,False


<iframe srcdoc="<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<style>
html,body{margin:0;height:100%;background:transparent;color:#e6e9f5;font:12px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;overflow:hidden}
#root{position:relative;height:620px;border-radius:16px;overflow:hidden;border:1px solid rgba(140,160,255,.2);
 background:radial-gradient(900px 480px at 15% 5%,rgba(99,102,241,.2),transparent 60%),radial-gradient(800px 480px at 95% 95%,rgba(34,211,238,.14),transparent 60%),#04050d}
canvas{position:absolute;inset:0;width:100%;height:100%;cursor:grab}
.glass{position:absolute;background:rgba(8,11,26,.7);backdrop-filter:blur(12px);border:1px solid rgba(140,160,255,.2);border-radius:10px;padding:7px 11px}
#legend{left:10px;bottom:10px;max-height:40%;overflow:auto}
.lg{display:flex;gap:8px;align-items:center;padding:2px 0;white-space:nowrap}
.lg i{width:9px;height:9px;border-radius:50%;box-shadow:0 0 8px currentColor}.lg b{font-weight:500;color:#fbbf24}
#buttons{right:10px;top:10px;display:flex;gap:6px;padding:4px}
button{background:none;border:0;color:#cbd2ee;cursor:pointer;font:inherit;padding:3px 9px;border-radius:6px}button:hover{background:rgba(255,255,255,.1)}
#tip{pointer-events:none;opacity:0;max-width:260px;transition:opacity .15s}#tip.on{opacity:1}#tip div{color:#8b93b8;margin-top:2px}
#empty{display:none;inset:0;align-items:center;justify-content:center;color:#8b93b8;position:absolute}
</style></head><body>
<div id="root"><canvas id="cv"></canvas>
<div id="legend" class="glass"></div>
<div id="buttons" class="glass"><button id="replay">↻ Replay</button><button id="fit">⤢ Fit</button></div>
<div id="tip" class="glass"></div><div id="empty">Index some documents to see the graph.</div></div>
<script>
(() => {
const D = {"nodes": [{"kind": "chunk", "doc": 0, "title": "Engineering Onboarding Guide", "n": 1, "snip": "# Engineering Onboarding Guide Welcome to Aurora Labs engineering. This guide covers your first 90 days. ## Week one On day one you receive your laptop and an onboarding buddy. By the end of the first", "status": "fresh", "x": -0.5282, "y": -0.1765}, {"kind": "chunk", "doc": 0, "title": "Engineering Onboarding Guide", "n": 2, "snip": "## Development environment We standardise on Python 3.12 managed with uv, Node 22 for front-end work, and Docker for local services. Clone the monorepo, run `make bootstrap`, and then `make test` to c", "status": "fresh", "x": 0.196, "y": 0.2768}, {"kind": "chunk", "doc": 0, "title": "Engineering Onboarding Guide", "n": 3, "snip": "## Code review Every change needs two approvals before it can merge, and at least one reviewer must come from outside your immediate team. Keep pull requests under 400 lines where possible. Reviewers ", "status": "fresh", "x": 0.0391, "y": -0.6215}, {"kind": "chunk", "doc": 0, "title": "Engineering Onboarding Guide", "n": 4, "snip": "## 30, 60 and 90 day goals By day 30 you ship your first feature behind a flag. By day 60 you own a small service or component. By day 90 you lead a design review and join the on-call rotation.", "status": "fresh", "x": -0.0317, "y": -0.2883}, {"kind": "chunk", "doc": 1, "title": "Incident Response Runbook", "n": 1, "snip": "# Incident Response Runbook This runbook describes how Aurora Labs detects, handles and learns from production incidents. ## Severity levels SEV1 means a full outage or data loss affecting many custom", "status": "fresh", "x": 0.0387, "y": -0.6225}, {"kind": "chunk", "doc": 1, "title": "Incident Response Runbook", "n": 2, "snip": "## Roles Every SEV1 and SEV2 incident has three roles. The incident commander coordinates the response and makes decisions. The communications lead posts updates to the status page and to affected cus", "status": "fresh", "x": 0.366, "y": -0.7589}, {"kind": "chunk", "doc": 1, "title": "Incident Response Runbook", "n": 3, "snip": "## Resolution and follow-up An incident is resolved only w

In [16]:
# Document health at a glance — edit a file in knowledge_base/ and re-run this plus kb.sync() to see re-indexing.
documents_table()


,document,file,chunks,status,detail
0,Engineering Onboarding Guide,engineering_onboarding.md,4,🟢 fresh,Last updated 2026-08-20.
1,Incident Response Runbook,incident_response_runbook.md,3,🟢 fresh,Last updated 2026-07-02.
2,RAG Primer,rag_primer.md,4,🟢 fresh,Last updated 2026-10-01.
3,Remote Work Policy,remote_work_policy.md,3,🟢 fresh,Last updated 2026-09-12.
4,Remote Work Policy (2022 edition),remote_work_policy_2022.md,1,🟣 superseded,Superseded by remote_work_policy.md.
5,Security Handbook,security_handbook.md,3,🔴 expired,Validity ended on 2026-06-30.
